In [30]:
# UnstructuredIO核心组件
from unstructured.partition.auto import partition
from typing import List
from unstructured.documents.elements import Element

# 使用partition函数自动检测文件类型并解析,默认strategy策略是auto，还会有fast策略，速度比image-to-text models的快100倍
elements: List[Element] = partition(filename="RAG评估.md", strategy="auto")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("==================")

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

print(type(elements[1]))
print(elements[4].text)

一.RAG效果评估的必要性
Title
{'category_depth': 0, 'languages': ['fra'], '_known_field_names': frozenset({'chunk_index', 'emphasized_text_tags', 'image_url', 'filename', 'enrichment_origins', 'page_name', 'detection_class_prob', 'cc_recipient', 'page_number', 'sent_from', 'filetype', 'link_texts', 'category_depth', 'data_source', 'email_message_id', 'image_mime_type', 'image_base64', 'image_path', 'header_footer_type', 'segment_end_seconds', 'links', 'num_carried_over_header_rows', 'link_urls', 'table_extraction_method', 'segment_start_seconds', 'languages', 'orig_elements', 'table_id', 'sent_to', 'table_as_cells', 'bcc_recipient', 'is_continuation', 'last_modified', 'routing_score', 'routing', 'attached_to_filename', 'detection_origin', 'text_as_html', 'file_directory', 'link_start_indexes', 'emphasized_text_contents', 'coordinates', 'key_value_pairs', 'subject', 'signature', 'is_extracted', 'url', 'parent_id'}), 'filename': 'RAG评估.md', 'filetype': 'text/markdown', 'last_modified': '2026-06-13

In [32]:
from unstructured.partition.auto import partition

# 🎯 只要加上这两个参数，让官方云端去帮你做高级版面分析
# 但是这个md太简单了，本地就能解析出来
elements = partition(
    filename="RAG评估.md",
    partition_via_api=True, # 走云端高精度模型
    api_key="RX1XicRJqOxKVUTCPAqIRyY8AmklRW"
)

for el in elements:
    print(f"[{el.category}] -> {el.text}")

[Title] -> 一.RAG效果评估的必要性
[Title] -> 二.RAG评估方法
[Title] -> 1.人工评估
[Title] -> 2.自动化评估
[Title] -> 3.LangSmith
[Title] -> 4.RAGAS
[Table] -> 维度 LangSmith RAGAS 核心定位 大模型应用的 集成开发平台 (调试、测试、评估、监控) 专门的RAG评估框架 ，用于量化RAG管道在不同组件层面上的性能 核心功能 提供全链路功能：应用 调试、测试、评估、监控 专注于评估 ，提供针对RAG的专用评估指标 评估方式 支持 自定义评估函数 和 基于参考答案的评估 (如精确匹配) ，以及 LLM即评委 等多种方式 提供一套 预设的、无需参考答案 的评估指标 ，可程序化计算 关键评估指标 支持广泛，取决于配置。可包括 精确匹配、工具调用准确性、自定义指标 等 忠实度、答案相关性、上下文精度、上下文召回率 等RAG核心指标 使用复杂度 相对较高，需要集成到开发流程中，配置数据集和评估器 相对较低，专注于评估，可通过几行代码对现有输入输出进行评估 数据需求 通常需要 构建包含输入和预期输出的测试数据集 无需参考答案 即可计算大部分核心指标
[Title] -> 三.评估指标


In [4]:
from typing import List, Dict, Any, Optional, Sequence
from pathlib import Path

# 自定义解析函数，支持任意类型的文件格式
def parse_file_with_unstructured(file_path: str):
    """
    使用UnstructuredIO解析单个文件

    Args:
        file_path: 文件路径

    Returns:
        Dict: 包含解析结果和统计信息的字典
    """
    print(f"\n 解析文件: {file_path}")

    try:
        # 使用partition函数自动检测文件类型并解析,默认strategy策略是auto，还会有fast策略，速度比image-to-text models的快100倍
        elements: List[Element] = partition(filename=file_path, strategy="auto")

        # 分析解析结果
        analysis = {
            "file_path": file_path,
            "file_extension": Path(file_path).suffix.lower(),
            "total_elements": len(elements),
            "element_types": {},
            "elements": elements,
            "text_content": "",
            "statistics": {}
        }

        # 统计元素类型
        for element in elements:
            element_type = type(element).__name__
            analysis["element_types"][element_type] = analysis["element_types"].get(element_type, 0) + 1

        # 提取文本内容
        text_parts = []

        for element in elements:
            if hasattr(element, 'text') and element.text:
                text_parts.append(element.text)

        analysis["text_content"] = "\n\n".join(text_parts)

        # 计算统计信息
        analysis["statistics"]["total_characters"] = len(analysis["text_content"])

        print(f"   解析完成")
        print(f"   元素总数: {analysis['total_elements']}")
        print(f"   元素类型: {analysis['element_types']}")
        print(f"   总字符数: {analysis['statistics']['total_characters']}")
        print(f"   文本内容: {analysis['text_content'][:200]} ")

    except Exception as e:
        print(f"文件解析失败: {e}")
        return {}

In [5]:
# %%markdown文档解析
parse_file_with_unstructured("RAG评估.md")


 解析文件: RAG评估.md
   解析完成
   元素总数: 8
   元素类型: {'Title': 7, 'Table': 1}
   总字符数: 474
   文本内容: 一.RAG效果评估的必要性

二.RAG评估方法

1.人工评估

2.自动化评估

3.LangSmith

4.RAGAS

维度 LangSmith RAGAS 核心定位 大模型应用的 集成开发平台 (调试、测试、评估、监控) 专门的RAG评估框架 ，用于量化RAG管道在不同组件层面上的性能 核心功能 提供全链路功能：应用 调试、测试、评估、监控 专注于评估 ，提供针对RAG的专用评估指标  


In [6]:
from unstructured.partition.md import partition_md
from typing import List
from unstructured.documents.elements import Element

# 使用partition_md函数检测markdown文件类型解析,include_page_breaks若希望在 Markdown 中标识页面断点（少见场景）
elements: List[Element] = partition_md(filename="RAG评估.md", languages=["zho"],include_page_breaks=True)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'category_depth': 0, 'languages': ['zho'], '_known_field_names': frozenset({'chunk_index', 'emphasized_text_tags', 'image_url', 'filename', 'enrichment_origins', 'page_name', 'detection_class_prob', 'cc_recipient', 'page_number', 'sent_from', 'filetype', 'link_texts', 'category_depth', 'data_source', 'email_message_id', 'image_mime_type', 'image_base64', 'image_path', 'header_footer_type', 'segment_end_seconds', 'links', 'num_carried_over_header_rows', 'link_urls', 'table_extraction_method', 'segment_start_seconds', 'languages', 'orig_elements', 'table_id', 'sent_to', 'table_as_cells', 'bcc_recipient', 'is_continuation', 'last_modified', 'routing_score', 'routing', 'attached_to_filename', 'detection_origin', 'text_as_html', 'file_directory', 'link_start_indexes', 'emphasized_text_contents', 'coordinates', 'key_value_pairs', 'subject', 'signature', 'is_extracted', 'url', 'parent_id'}), 'filename': 'RAG评估.md', 'filetype': 'text/markdown', 'last_modified': '2026-06-13T17:21:40'}
一.RAG效果评

In [22]:
# html文档解析
parse_file_with_unstructured("html-tags-decode.html")


 解析文件: html-tags-decode.html
   解析完成
   元素总数: 4
   元素类型: {'Title': 1, 'NarrativeText': 1, 'Text': 2}
   总字符数: 232
   文本内容: 识别和解析HTML标签

HTML tags (filter) decode, You can increase safety by filtering the danger label.

注：虽然此功能能极大地扩展 Markdown 语法，但也面临着安全上的风险，所以默认是不开启的。

Update: 可以通过设置 `settings.htmlDecode = "style,script,if 


In [23]:
from unstructured.partition.html import partition_html
from typing import List
from unstructured.documents.elements import Element

# 使用partition_html函数检测html网页类型解析
elements = partition_html(url="https://docs.unstructured.io/welcome",
                          headers={"User-Agent":"MyBot"},
                          ssl_verify=False,
                          include_page_breaks=False,
                          encoding="utf-8")
#elements: List[Element] = partition_html(url="https://docs.unstructured.io/welcome", languages=["zho"])

# 元素的元数据
print(elements[1].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[1].text)
print("===========================")

# 元素的类型
print(elements[1].category)
print("===========================")

/root/miniconda3/envs/my_rag/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'docs.unstructured.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'link_texts': ['Learn about Unstructured’s product offerings'], 'link_urls': ['/about'], 'languages': ['fra'], '_known_field_names': frozenset({'chunk_index', 'emphasized_text_tags', 'image_url', 'filename', 'enrichment_origins', 'page_name', 'detection_class_prob', 'cc_recipient', 'page_number', 'sent_from', 'filetype', 'link_texts', 'category_depth', 'data_source', 'email_message_id', 'image_mime_type', 'image_base64', 'image_path', 'header_footer_type', 'segment_end_seconds', 'links', 'num_carried_over_header_rows', 'link_urls', 'table_extraction_method', 'segment_start_seconds', 'languages', 'orig_elements', 'table_id', 'sent_to', 'table_as_cells', 'bcc_recipient', 'is_continuation', 'last_modified', 'routing_score', 'routing', 'attached_to_filename', 'detection_origin', 'text_as_html', 'file_directory', 'link_start_indexes', 'emphasized_text_contents', 'coordinates', 'key_value_pairs', 'subject', 'signature', 'is_extracted', 'url', 'parent_id'}), 'filetype': 'text/html', 'url': '

In [24]:
parse_file_with_unstructured("销售数据统计.xlsx")

No features in text.
No features in text.



 解析文件: 销售数据统计.xlsx
   解析完成
   元素总数: 1
   元素类型: {'Table': 1}
   总字符数: 1940
   文本内容: 日期 销售人员ID 销量 销售金额 10/20/2018 3 100 300 10/14/2018 4 100 100 12/20/2018 5 400 1200 10/23/2018 2 300 900 11/16/2018 3 100 100 10/30/2018 5 400 800 11/5/2018 5 100 200 12/28/2018 1 100 300 11/20/2018 1 1 


In [ ]:
from unstructured.partition.xlsx import partition_xlsx
from typing import List
from unstructured.documents.elements import Element

# 使用partition_xlsx函数检测excel文件类型并解析
elements: List[Element] = partition_xlsx(filename="销售数据统计.xlsx", languages=["zho"])

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

In [13]:
# 重新配置一个干净的、绝不复读的单管道输出
logging.basicConfig(level=logging.INFO, format='%(message)s')
parse_file_with_unstructured("训练数据.csv")


 解析文件: 训练数据.csv
   解析完成
   元素总数: 1
   元素类型: {'Table': 1}
   总字符数: 186191
   文本内容: months_as_customer age policy_number policy_bind_date policy_state policy_csl policy_deductable policy_annual_premium umbrella_limit insured_zip insured_sex insured_education_level insured_occupation  


In [14]:
from unstructured.partition.csv import partition_csv
from typing import List
from unstructured.documents.elements import Element

# 使用partition_csv函数检测csv文件类型并解析
elements = partition_csv(filename="训练数据.csv", encoding="utf-8")

# 元素的元数据
#print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

months_as_customer age policy_number policy_bind_date policy_state policy_csl policy_deductable policy_annual_premium umbrella_limit insured_zip insured_sex insured_education_level insured_occupation insured_hobbies insured_relationship capital-gains capital-loss incident_date incident_type collision_type incident_severity authorities_contacted incident_state incident_city incident_location incide
Table


In [17]:
# word文档解析
parse_file_with_unstructured("数组.docx")


 解析文件: 数组.docx
   解析完成
   元素总数: 153
   元素类型: {'Title': 21, 'NarrativeText': 14, 'Text': 43, 'ListItem': 75}
   总字符数: 7656
   文本内容: 1. 数组简介 

1.1 概述

我们之前学习的变量或者是常量, 只能用来存储一个数据, 例如: 存储一个整数, 小数或者字符串等. 如果需要同时存储多个同类型的数据, 用变量或者常量来实现的话, 非常的繁琐. 针对于这种情况, 我们就可以通过数组来实现了.

例如: 假设某公司有50名员工, 现在需要统计该公司员工的工资情况, 例如计算平均工资、获取最高工资等。针对于这个需求，如果用前面所学的 


In [18]:
from unstructured.partition.docx import partition_docx
from unstructured.partition.doc import partition_doc
from typing import List
from unstructured.documents.elements import Element

# 使用partition_docx函数检测word文件类型并解析，include_page_breaks当文档支持 “分页” 时，以标识不同页的边界
elements = partition_docx(filename="数组.docx", encoding="utf-8", include_page_breaks=True)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'category_depth': 2, 'filename': '数组.docx', 'last_modified': '2026-06-13T21:13:41', 'languages': ['fra'], '_known_field_names': frozenset({'chunk_index', 'emphasized_text_tags', 'image_url', 'filename', 'enrichment_origins', 'page_name', 'detection_class_prob', 'cc_recipient', 'page_number', 'sent_from', 'filetype', 'link_texts', 'category_depth', 'data_source', 'email_message_id', 'image_mime_type', 'image_base64', 'image_path', 'header_footer_type', 'segment_end_seconds', 'links', 'num_carried_over_header_rows', 'link_urls', 'table_extraction_method', 'segment_start_seconds', 'languages', 'orig_elements', 'table_id', 'sent_to', 'table_as_cells', 'bcc_recipient', 'is_continuation', 'last_modified', 'routing_score', 'routing', 'attached_to_filename', 'detection_origin', 'text_as_html', 'file_directory', 'link_start_indexes', 'emphasized_text_contents', 'coordinates', 'key_value_pairs', 'subject', 'signature', 'is_extracted', 'url', 'parent_id'}), 'filetype': 'application/vnd.openxmlfo

### PDF解析

In [42]:
parse_file_with_unstructured("PDF解析截图.png")


 解析文件: PDF解析截图.png
文件解析失败: libGL.so.1: cannot open shared object file: No such file or directory


{}

In [ ]:
from unstructured.partition.pdf import partition_pdf
from typing import List
from unstructured.documents.elements import Element
import io

print("🚀 启动【内存流绕道模式】，正在将 PDF 转化为二进制数据...")

# 1. 物理绕过：不在 partition_pdf 内部传 filename，而是我们自己用 Python 把文件读进内存
with open("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf", "rb") as f:
    pdf_bytes = f.read()

# 2. 将纯二进制数据包装成标准的内存文件流
file_like_object = io.BytesIO(pdf_bytes)

print("📡 正在跨过本地探查，直接向云端 API 发送纯净网络请求...")

# 使用partition_pdf函数检测pdf类型并解析

elements = partition_pdf(filename=file_like_object,
                         partition_via_api=True,          # 🎯 核心：发给云端去解析，不吃本地 2G 内存！
                         api_key="RX1XicRJqOxKVUTCPAqIRyY8AmklRW",
                         strategy="hi_res", # 使用hi_res模式进行高精度解析
                         extract_images_in_pdf=False, # 提取pdf中的图片
                         #extract_image_block_types=["Table","Image"], # 提取表格和图片
                         #extract_image_block_output_dir="./images", # 保存图片到images目录
                         languages=["eng","zho"],
                         split_pdf_page=False, # 大文件分块处理，优化性能
                         infer_table_structure=False, # 是否尝试推断表格结构，会下载一个视觉目标检测的 Transformer 模型
                         include_page_breaks=True,
                         request_timeout=600) # 是否包含页码信息


# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

In [6]:
import requests
import json

print("📡 正在跨过第三方框架，直接向云端 API 节点发起直连...")

url = "https://api.unstructured.io/general/v1/general"
headers = {
    "accept": "application/json",
    "unstructured-api-key": "RX1XicRJqOxKVUTCPAqIRyY8AmklRW" # 你的免费 API Key
}

# 1. 组装发给云端的参数（对应你的高精度、不吃本地内存的配置）
data = {
    "strategy": "hi_res",
    "languages": ["eng", "zho"],
    "split_pdf_page": "false",
    "infer_table_structure": "false",
    "extract_images_in_pdf": "false",
    "include_page_breaks": "true"
}

# 2. 以标准的二进制表单形式物理打开本地 PDF 文件
files = {
    "files": open("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf", "rb")
}

try:
    # 3. 发送网络请求，设置 600 秒超长超时续命
    response = requests.post(url, headers=headers, data=data, files=files, timeout=600)
    
    if response.status_code == 200:
        elements = response.json()
        print(f"🎉 成功通关！云端高精度解析返回了 {len(elements)} 个元素数据！")
        
        # 演示打印：看看前三个元素的类型和文本内容
        for index, el in enumerate(elements[:3]):
            print(f"\n元素 [{index}] 类型: {el.get('type')}")
            print(f"文本内容: {el.get('text')[:150]}...")
            
    else:
        print(f"❌ 云端返回错误，状态码: {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"💥 网络物理中断: {e}")

📡 正在跨过第三方框架，直接向云端 API 节点发起直连...
💥 网络物理中断: HTTPSConnectionPool(host='api.unstructured.io', port=443): Max retries exceeded with url: /general/v1/general (Caused by NameResolutionError("HTTPSConnection(host='api.unstructured.io', port=443): Failed to resolve 'api.unstructured.io' ([Errno -2] Name or service not known)"))


In [21]:
from llama_index.core import SimpleDirectoryReader
from llama_parse import LlamaParse

# 如果文档结构复杂，优先使用 LlamaParse
# parser = LlamaParse(api_key="YOUR_LLAMA_CLOUD_API_KEY")
# documents = parser.load_data("sample.pdf")

# 或者使用简单读取器
documents = SimpleDirectoryReader(input_files=["RAG评估.md"]).load_data()

print(documents[0])
print("===========================")
print(documents[0].metadata)
print("===========================")
print(documents[0].text)
print("===========================")

Doc ID: 8686322d-3a8d-4d3b-9713-038dd7896c57
Text: <h1 id="A9atq">一.RAG效果评估的必要性</h1> + <font style="color:rgb(0, 0,
0);">评估出RAG对大模型能力改善的程度</font> + <font style="color:rgb(0, 0,
0);">RAG优化过程，通过评估可以知道改善的方向和参数调整的程度</font>  <h1
id="HCceE">二.RAG评估方法</h1> <h2 id="MyoUa">1.人工评估</h2> + <font
style="color:rgb(0, 0, 0);">最Low的方式是进行人工评估：邀请专家或人工评估员对RAG生成的结果进行评估。他们可
以根据预先定义的标准对生成的答案进行质量评估，如准确性、连贯性、相关性等。这种评估方法...
{'file_path': 'RAG评估.md', 'file_name': 'RAG评估.md', 'file_type': 'text/markdown', 'file_size': 5828, 'creation_date': '2026-06-13', 'last_modified_date': '2026-06-13'}
<h1 id="A9atq">一.RAG效果评估的必要性</h1>
+ <font style="color:rgb(0, 0, 0);">评估出RAG对大模型能力改善的程度</font>
+ <font style="color:rgb(0, 0, 0);">RAG优化过程，通过评估可以知道改善的方向和参数调整的程度</font>

<h1 id="HCceE">二.RAG评估方法</h1>
<h2 id="MyoUa">1.人工评估</h2>
+ <font style="color:rgb(0, 0, 0);">最Low的方式是进行人工评估：邀请专家或人工评估员对RAG生成的结果进行评估。他们可以根据预先定义的标准对生成的答案进行质量评估，如准确性、连贯性、相关性等。这种评估方法可以提供高质量的反馈，但可能会消耗大量的时间和人力资源。</font>

<h2 id="hJpzW">2.自动化评估</h2>
+ <font style="col

In [24]:
from llama_index.readers.file.unstructured import UnstructuredReader
from pathlib import Path

reader = UnstructuredReader()
# 如果要控制是否启用 hi_res ，load_data中可以传参
documents = reader.load_data(file=Path("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf"))

print("打印列表长度：" + str(len(documents)))
print("==================================")
print("打印解析的文本内容：" + documents[0].text[:100])
print("==================================")
print("打印元数据信息：" + str(documents[0].metadata))

2026-06-14 23:24:30,178 - WARNING - No languages specified, defaulting to English.
2026-06-14 23:24:30,198 - WARNING - 'doc_id' is deprecated and 'id_' will be used instead


打印列表长度：1
打印解析的文本内容：证 券 研 究 报 告

行 业 研 究

行 业 点 评

海外科技巨头持续发力 AI，龙头公司中报业绩亮眼

——AI 行业点评报告

◼ 核心观点 海外 AI 视角：（1）英伟达推出 B200A
打印元数据信息：{'coordinates': '{"points": [[6.84, 80.35964000000001], [6.84, 163.39963999999975], [20.88, 163.39963999999975], [20.88, 80.35964000000001]], "system": "PixelSpace", "layout_width": 595.32, "layout_height": 841.92}', 'filename': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'last_modified': '2026-06-14T12:49:01', 'page_number': 1, 'languages': '["zho"]', 'filetype': 'application/pdf'}


In [ ]:
from unstructured.partition.auto import partition
# 使用LlamaIndex的Document对象，将解析后的元素转换为Document对象
from llama_index.core import Document

# 使用partition函数自动检测文件类型并解析
elements = partition(
    filename="甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
    strategy="hi_res",
    split_pdf_page=True,
    infer_table_structure=True,
    languages=["eng","chi_sim"])

# 将解析后的元素转换为Document对象
docs = [
    Document(text=e.text,
             metadata={"source":"甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
                       "type": e.category})
    for e in elements]


In [26]:
from llama_index.core import Document
# 导入SentenceSplitter句子分割器
from llama_index.core.node_parser import SentenceSplitter

# 创建Document并设置元数据
doc = Document(
    text="这是一份关于RAG技术的文档...",
    metadata={
        "file_name": "rag_guide.pdf",
        "category": "技术文档",
        "author": "AI研究团队",
        "created_date": "2023-11-15"
    }
)

# 从Document创建Node时，元数据会自动传播
splitter = SentenceSplitter()
nodes = splitter.get_nodes_from_documents([doc])

# 每个node都会继承doc的metadata
nodes[0].metadata

{'file_name': 'rag_guide.pdf',
 'category': '技术文档',
 'author': 'AI研究团队',
 'created_date': '2023-11-15'}